# Results Analysis: Final Plots, Summary Tables, Qualitative Examples

Aggregates results from **notebook 05** (poisoning sweep) and **notebook 06** (prompt ablation +
self-consistency) into final figures and tables for the report.

**No new API calls**: all data is re-computed from the LLM response cache.

Attribution map:
- Poisoning sweep design: Zhou et al. 2024 (Robustness dimension, §2–3).
- Factuality metrics (accuracy, macro-F1, hallucination rate): Zhou et al. 2024.
- Self-consistency metric: Wang et al. 2022 (cited in Zhou 2024 §2.1).
- Prompt templates: Zhou et al. 2024 (CoT, vigilant) + Singal et al. 2024 (standard).
- RAG pipeline + FEVER evaluation: Lewis et al. 2020.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

SEED            = cfg["seed"]
K               = cfg["retrieval"]["k"]
DISTRACTOR_POOL = cfg["retrieval"]["distractor_pool_size"]
EMBEDDING_MODEL = cfg["retrieval"]["embedding_model"]
EMB_CACHE       = os.path.join("..", cfg["cache"]["dir"], cfg["cache"]["embeddings_subdir"])
LLM_CACHE       = os.path.join("..", cfg["cache"]["dir"], cfg["cache"]["llm_subdir"])

MODELS          = cfg["models"]["available"]
PROMPT_TYPES    = ["standard", "chain_of_thought", "vigilant"]
POISON_RATES    = [0.0, 0.25, 0.5, 0.75, 1.0]
TEMPERATURE     = cfg["models"]["temperature"]
TEMP_SC         = cfg["models"]["temperature_consistency"]
N_EXAMPLES      = cfg["evaluation"]["n_examples"]
N_SC            = cfg["evaluation"]["self_consistency_subset"]
SC_RUNS         = cfg["evaluation"]["self_consistency_runs"]


from nb_style import MODEL_LABELS, MODEL_COLORS, MODEL_LS, MODEL_MARKERS, PROMPT_COLORS

os.makedirs("../figures", exist_ok=True)
print("Config loaded.")

## 1. Reproduce all data from cache

Re-run the scorer for every condition. All calls hit the disk cache: no new API cost.

In [ ]:
from src.data.fever_loader import load_fever
from src.data.poisoner import poison_dataset
from src.retrieval.embedder import Embedder
from src.retrieval.retriever import Retriever
from src.generation.llm_client import HuggingFaceClient
from src.evaluation.scorer import run as run_scorer

all_examples = load_fever("../" + cfg["dataset"]["fever_dev"])
examples     = all_examples[:N_EXAMPLES]
examples_sc  = all_examples[:N_SC]

embedder = Embedder(model_name=EMBEDDING_MODEL, cache_dir=EMB_CACHE, device="cpu")


def build_llm(model: str, temperature: float = 0.0):
    return HuggingFaceClient(model=model, temperature=temperature, cache_dir=LLM_CACHE)


# --- Phase A: full poisoning sweep (nb05 replica) ---
rows_main = []
for model_name in MODELS:
    llm = build_llm(model_name, TEMPERATURE)
    with llm:
        for pr in POISON_RATES:
            poisoned = poison_dataset(examples, poison_rate=pr, seed=SEED) if pr > 0 else examples
            for pt in PROMPT_TYPES:
                retriever = Retriever(embedder=embedder, k=K)
                m = run_scorer(poisoned, retriever, llm, prompt_type=pt,
                               distractor_pool_size=DISTRACTOR_POOL, seed=SEED,
                               self_consistency_runs=1)
                rows_main.append({"model": model_name, "poison_rate": pr, "prompt_type": pt, **m})
    print(f"  {MODEL_LABELS[model_name]} sweep done")

df_main = pd.DataFrame(rows_main)
print(f"Main sweep: {len(df_main)} conditions (all cached).")

# --- Phase B: self-consistency at poison_rate=0.5 (nb06 replica) ---
rows_sc = []
poisoned_sc = poison_dataset(examples_sc, poison_rate=0.5, seed=SEED)
for model_name in MODELS:
    llm = build_llm(model_name, TEMP_SC)
    with llm:
        for pt in PROMPT_TYPES:
            retriever = Retriever(embedder=embedder, k=K)
            m = run_scorer(poisoned_sc, retriever, llm, prompt_type=pt,
                           distractor_pool_size=DISTRACTOR_POOL, seed=SEED,
                           self_consistency_runs=SC_RUNS)
            rows_sc.append({"model": model_name, "prompt_type": pt,
                            "self_consistency": m["self_consistency"]})
    print(f"  {MODEL_LABELS[model_name]} self-consistency done")

df_sc = pd.DataFrame(rows_sc)
embedder.close()
print(f"Self-consistency: {len(df_sc)} conditions.")

## 2. Grand Summary Table

All metrics across every condition, sorted by model and poison rate.

In [ ]:
# Merge self-consistency (measured at poison_rate=0.5) into the main table
df = df_main.merge(df_sc, on=["model", "prompt_type"], how="left")

# Add short model name
df["model_short"] = df["model"].map(MODEL_LABELS)

# Display full table
display_cols = ["model_short", "poison_rate", "prompt_type",
                "accuracy", "macro_f1", "hallucination_rate", "precision_at_k", "self_consistency"]
table = df[display_cols].copy()
table.columns = ["Model", "Poison Rate", "Prompt", "Accuracy", "Macro-F1",
                  "Hallucination", "P@k", "Self-Consistency"]
print(f"=== Grand Summary Table (N={N_EXAMPLES}, k={K}) ===")
print(f"    Self-consistency measured at poison_rate=0.5, temp={TEMP_SC}, sc_runs={SC_RUNS}, N={N_SC}")
print()
print(table.to_string(index=False, float_format="{:.3f}".format))

## 3. Accuracy Degradation: Robustness Delta

How much accuracy each condition loses relative to its own clean baseline (poison_rate=0%).
Smaller delta = more robust. Attribution: Zhou et al. 2024: Robustness dimension.

In [ ]:
# Compute delta from clean baseline for each (model, prompt_type) pair
baselines = df_main[df_main["poison_rate"] == 0.0].set_index(["model", "prompt_type"])["accuracy"]
df_main["acc_delta"] = df_main.apply(
    lambda r: r["accuracy"] - baselines.loc[(r["model"], r["prompt_type"])], axis=1
)

fig, axes = plt.subplots(1, len(MODELS), figsize=(16, 4), sharey=True)
fig.suptitle(f"Accuracy Delta from Clean Baseline (k={K}, N={N_EXAMPLES})", fontsize=13, y=1.02)

for ax, model in zip(axes, MODELS):
    for pt in PROMPT_TYPES:
        sub = df_main[(df_main["model"] == model) & (df_main["prompt_type"] == pt)]
        sub = sub.sort_values("poison_rate")
        ax.plot(sub["poison_rate"], sub["acc_delta"],
                marker="o", linewidth=2, markersize=5,
                color=PROMPT_COLORS[pt], linestyle="-", label=pt)
    ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
    ax.set_title(MODEL_LABELS[model], fontsize=10)
    ax.set_xlabel("Poison Rate", fontsize=8)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle=":", alpha=0.5)

axes[0].set_ylabel("Accuracy − Accuracy@0%")
handles = [mpatches.Patch(facecolor=PROMPT_COLORS[pt], label=pt) for pt in PROMPT_TYPES]
fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.savefig("../figures/05_accuracy_delta.png", dpi=150, bbox_inches="tight")
print("Saved -> figures/05_accuracy_delta.png")
plt.show()

## 4. Hallucination vs Accuracy: Scatter (all conditions)

Each dot is one (model, prompt, poison_rate) condition. Ideal = top-left (high accuracy, low hallucination).
Attribution: Zhou et al. 2024: Factuality dimension.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for model in MODELS:
    for pt in PROMPT_TYPES:
        sub = df_main[(df_main["model"] == model) & (df_main["prompt_type"] == pt)]
        ax.scatter(sub["hallucination_rate"], sub["accuracy"],
                   c=PROMPT_COLORS[pt], marker=MODEL_MARKERS[model],
                   s=70, alpha=0.85, edgecolors="white", linewidth=0.5)

ax.set_xlabel("Hallucination Rate (lower is better)")
ax.set_ylabel("Accuracy (higher is better)")
ax.set_title(f"Accuracy vs Hallucination Rate: all conditions (k={K})", fontsize=13, pad=10)
ax.set_xlim(-0.02, 0.85)
ax.set_ylim(0.25, 0.85)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(linestyle=":", alpha=0.4)
ax.annotate("ideal", xy=(0.02, 0.80), fontsize=10, fontstyle="italic", color="grey")

prompt_handles = [mpatches.Patch(facecolor=PROMPT_COLORS[pt], label=pt) for pt in PROMPT_TYPES]
model_handles  = [plt.Line2D([0], [0], marker=MODEL_MARKERS[m], color="grey",
                              linestyle="None", markersize=7, label=MODEL_LABELS[m])
                  for m in MODELS]
ax.legend(handles=prompt_handles + model_handles, fontsize=7, loc="lower left", ncol=2, framealpha=0.7)

plt.tight_layout()
plt.savefig("../figures/05_hallucination_vs_accuracy.png", dpi=150, bbox_inches="tight")
print("Saved -> figures/05_hallucination_vs_accuracy.png")
plt.show()

## 5. Heatmap: Mean Accuracy by Prompt x Model (averaged over all poison rates)

Collapses the poison-rate dimension to show the overall ranking of conditions.

In [ ]:
# 3-metric heatmap grid: accuracy, macro-f1, hallucination
metrics_cfg = [
    ("accuracy",           "Accuracy",           "RdYlGn",  True),
    ("macro_f1",           "Macro-F1",            "RdYlGn",  True),
    ("hallucination_rate", "Hallucination Rate",  "YlOrRd",  False),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 3.5))

for ax, (col, title, cmap, higher_better) in zip(axes, metrics_cfg):
    pivot = df_main.pivot_table(index="prompt_type", columns="model",
                                values=col, aggfunc="mean")
    pivot = pivot.reindex(PROMPT_TYPES)
    pivot.columns = [MODEL_LABELS[c] for c in pivot.columns]

    im = ax.imshow(pivot.values, cmap=cmap, vmin=0, vmax=1 if "rate" in col else pivot.values.max() * 1.15,
                   aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=8, rotation=20, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)

    arrow = "higher better" if higher_better else "lower better"
    ax.set_title(f"{title}\n({arrow})", fontsize=10, pad=6)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                    fontsize=8, fontweight="bold",
                    color="white" if (higher_better and v > 0.7) or (not higher_better and v > 0.4) else "black")

fig.suptitle(f"Mean Metrics across Poison Rates (k={K}, N={N_EXAMPLES})", fontsize=13, y=1.04)
plt.tight_layout()
plt.savefig("../figures/05_metric_heatmaps.png", dpi=150, bbox_inches="tight")
print("Saved -> figures/05_metric_heatmaps.png")
plt.show()

## 6. Non-Monotonic Accuracy: the U-Shaped Recovery

Key finding: accuracy does not degrade linearly with poison rate. At 100% poisoning, precision@k=0
(no gold passages retrieved), so models fall back on parametric knowledge (Lewis et al. 2020).
The most dangerous regime is 25-75% where models partially trust wrong context.

In [ ]:
# Average accuracy across prompts per model
avg_by_model = df_main.groupby(["model", "poison_rate"])["accuracy"].mean().reset_index()

# Figure 1: mean accuracy per model (U-shape overview)
fig1, ax1 = plt.subplots(figsize=(7, 5))
for model in MODELS:
    sub = avg_by_model[avg_by_model["model"] == model].sort_values("poison_rate")
    ax1.plot(sub["poison_rate"], sub["accuracy"],
             marker="o", linewidth=2.5, markersize=7,
             linestyle=MODEL_LS[model], color=MODEL_COLORS[model],
             label=MODEL_LABELS[model])
ax1.set_title("Mean Accuracy (avg over prompts)", fontsize=12, pad=8)
ax1.set_xlabel("Poison Rate")
ax1.set_ylabel("Accuracy")
ax1.set_xlim(-0.02, 1.02)
ax1.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax1.legend(fontsize=8)
ax1.spines[["top", "right"]].set_visible(False)
ax1.grid(axis="y", linestyle=":", alpha=0.5)
ax1.axvspan(0.20, 0.80, alpha=0.08, color="red")
ax1.annotate("parametric\nknowledge\nfallback", xy=(0.95, 0.62), fontsize=8,
             fontstyle="italic", color="grey", ha="center")
fig1.tight_layout()
fig1.savefig("../figures/05_non_monotonic_mean.png", dpi=150, bbox_inches="tight")
print("Saved -> figures/05_non_monotonic_mean.png")
plt.show()

# Figure 2: per-condition accuracy: faceted by model (1×5)
fig2, axes2 = plt.subplots(1, len(MODELS), figsize=(18, 4), sharey=True)
fig2.suptitle(f"Per-Condition Accuracy Curves (k={K}, N={N_EXAMPLES})", fontsize=13, y=1.02)
for ax, model in zip(axes2, MODELS):
    for pt in PROMPT_TYPES:
        sub = df_main[(df_main["model"] == model) & (df_main["prompt_type"] == pt)]
        sub = sub.sort_values("poison_rate")
        ax.plot(sub["poison_rate"], sub["accuracy"],
                marker="o", linewidth=1.8, markersize=4,
                color=PROMPT_COLORS[pt], linestyle="-", label=pt)
    ax.axvspan(0.20, 0.80, alpha=0.08, color="red")
    ax.set_title(MODEL_LABELS[model], fontsize=10)
    ax.set_xlabel("Poison Rate", fontsize=8)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle=":", alpha=0.5)
axes2[0].set_ylabel("Accuracy")
handles = [mpatches.Patch(facecolor=PROMPT_COLORS[pt], label=pt) for pt in PROMPT_TYPES]
fig2.legend(handles=handles, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.08))
fig2.tight_layout()
fig2.savefig("../figures/05_non_monotonic_accuracy.png", dpi=150, bbox_inches="tight")
print("Saved -> figures/05_non_monotonic_accuracy.png")
plt.show()

## 7. Retrieval Quality vs. Accuracy: Correlation Analysis

Does a weaker retriever always mean a less accurate generator, or does prompt design mediate this?  
Each point is one `(model, prompt_type, poison_rate)` condition from the poisoning sweep.

In [ ]:
from src.evaluation.metrics import retrieval_accuracy_correlation

fig, ax = plt.subplots(figsize=(8, 6))

for pt in PROMPT_TYPES:
    sub = df_main[df_main['prompt_type'] == pt]
    for model in MODELS:
        s = sub[sub['model'] == model]
        ax.scatter(
            s['precision_at_k'], s['accuracy'],
            c=PROMPT_COLORS[pt], marker=MODEL_MARKERS[model],
            s=60, alpha=0.8, edgecolors='white', linewidth=0.4,
        )
    # Trend line per prompt type
    x_all = sub['precision_at_k'].values
    y_all = sub['accuracy'].values
    m_fit, b_fit = np.polyfit(x_all, y_all, 1)
    x_line = np.linspace(x_all.min(), x_all.max(), 50)
    ax.plot(x_line, m_fit * x_line + b_fit, color=PROMPT_COLORS[pt],
            linewidth=1.8, label=pt.replace('_', ' ').title())

ax.set_xlabel('Precision@k')
ax.set_ylabel('Accuracy')
ax.set_title('Retrieval Quality vs. Accuracy (all conditions)')
prompt_legend = ax.legend(title='Prompt type', loc='upper left')
model_handles = [
    plt.scatter([], [], marker=MODEL_MARKERS[m], color='grey', s=45, label=MODEL_LABELS[m])
    for m in MODELS
]
ax.add_artist(ax.legend(handles=model_handles, title='Model', loc='lower right', fontsize=8))
ax.add_artist(prompt_legend)
plt.tight_layout()
plt.savefig('../figures/07_retrieval_accuracy_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Correlation statistics ---
global_r = retrieval_accuracy_correlation(
    df_main['precision_at_k'].tolist(),
    df_main['accuracy'].tolist(),
)
print(f"Global  Pearson r={global_r['pearson_r']:.3f}  p={global_r['pearson_p']:.4f}")
print(f"Global Spearman r={global_r['spearman_r']:.3f}  p={global_r['spearman_p']:.4f}")
print()

rows_corr = []
for pt in PROMPT_TYPES:
    sub = df_main[df_main['prompt_type'] == pt]
    r = retrieval_accuracy_correlation(
        sub['precision_at_k'].tolist(), sub['accuracy'].tolist()
    )
    rows_corr.append({
        'Prompt':     pt,
        'Pearson r':  round(r['pearson_r'],  3),
        'Pearson p':  round(r['pearson_p'],  4),
        'Spearman r': round(r['spearman_r'], 3),
        'Spearman p': round(r['spearman_p'], 4),
    })

df_corr = pd.DataFrame(rows_corr).set_index('Prompt')
print(df_corr.to_string())

**Findings** *(fill in after running)*: across all conditions precision@k and accuracy show a [positive/negative] correlation (global Pearson r = ___, p ___).  
The vigilant prompt type shows a [weaker/stronger] coupling (r = ___) than standard (r = ___), suggesting that explicit reasoning instructions [partially/do not] decouple generation accuracy from retrieval quality.  
The weakest correlation is observed for the chain-of-thought prompt, consistent with the U-shaped recovery pattern: the model's own reasoning chain compensates for poisoned context at extreme poison rates.

## 8. Qualitative Examples

Show specific claims where the models' predictions diverge between clean (0%) and poisoned (50%)
conditions. Helps illustrate *how* poisoning changes model behaviour.

In [ ]:
from src.retrieval.corpus import build_corpus
from src.generation.prompts import format_prompt
from src.generation.parser import extract_label

# Re-open LLM clients (cached, no cost) to get per-example predictions
# Compare: standard prompt, claude, poison_rate=0.0 vs 0.5
QUAL_MODEL = MODELS[0]   # claude
QUAL_PT    = "standard"
QUAL_N     = 20  # show first 20 examples
QUAL_PR    = [0.0, 0.5]

# Use the full N_EXAMPLES pool (same as nb03) so poison_dataset and build_corpus
# produce identical prompts -> cache hit. Only iterate over first QUAL_N.
embedder = Embedder(model_name=EMBEDDING_MODEL, cache_dir=EMB_CACHE, device="cpu")
llm = build_llm(QUAL_MODEL, TEMPERATURE)

qual_rows = []
with llm:
    for pr in QUAL_PR:
        poisoned = poison_dataset(examples, poison_rate=pr, seed=SEED) if pr > 0 else examples
        for i, ex in enumerate(poisoned[:QUAL_N]):
            corpus = build_corpus(ex, poisoned, distractor_pool_size=DISTRACTOR_POOL,
                                  seed=SEED + i, example_index=i)
            retriever = Retriever(embedder=embedder, k=K)
            retriever.build(corpus)
            passages = retriever.retrieve(ex["claim"])
            prompt = format_prompt(ex["claim"], passages, QUAL_PT)
            response = llm.complete(prompt, max_tokens=64)
            pred = extract_label(response)
            qual_rows.append({
                "idx": i, "claim": ex["claim"], "gold": ex["label"],
                "poison_rate": pr, "prediction": pred,
                "passages_preview": " | ".join(p[:60] + "..." for p in passages[:3]),
            })

embedder.close()
df_qual = pd.DataFrame(qual_rows)

# Pivot: show claim + gold + prediction at 0% vs 50%
clean  = df_qual[df_qual["poison_rate"] == 0.0].set_index("idx")
poison = df_qual[df_qual["poison_rate"] == 0.5].set_index("idx")

comparison = pd.DataFrame({
    "Claim": clean["claim"].str[:80],
    "Gold": clean["gold"],
    "Pred@0%": clean["prediction"],
    "Pred@50%": poison["prediction"],
    "Flipped": clean["prediction"] != poison["prediction"],
})

print(f"=== Qualitative Comparison: {MODEL_LABELS[QUAL_MODEL]} / {QUAL_PT} (first {QUAL_N} claims) ===\n")
print(comparison.to_string())
print(f"\nFlipped predictions: {comparison['Flipped'].sum()} / {len(comparison)}")

In [ ]:
# Focus on flipped examples: show claim, gold, and what changed
flipped = comparison[comparison["Flipped"]]
if len(flipped) > 0:
    print(f"=== Flipped Predictions (clean -> poisoned) ===\n")
    for idx, row in flipped.iterrows():
        correct_clean = "correct" if row["Pred@0%"] == row["Gold"] else "WRONG"
        correct_poison = "correct" if row["Pred@50%"] == row["Gold"] else "WRONG"
        print(f"  [{idx}] {row['Claim']}")
        print(f"       Gold: {row['Gold']}")
        print(f"       @0%:  {row['Pred@0%']} ({correct_clean})")
        print(f"       @50%: {row['Pred@50%']} ({correct_poison})")
        print()
else:
    print("No predictions flipped in this subset.")

## 9. NEI Analysis: Hallucination Breakdown by Gold Label

Hallucination rate only applies to NOT ENOUGH INFO claims. Here we break down how
each condition handles NEI vs factual (SUPPORTS/REFUTES) claims separately.
Attribution: Zhou et al. 2024: Factuality dimension, hallucination quantification.

In [ ]:
# Per-example predictions for ALL conditions at 0% and 50% poisoning
# to compute per-label accuracy
from src.evaluation.metrics import accuracy as compute_accuracy

embedder = Embedder(model_name=EMBEDDING_MODEL, cache_dir=EMB_CACHE, device="cpu")

label_rows = []
for pr in [0.0, 0.5]:
    poisoned = poison_dataset(examples, poison_rate=pr, seed=SEED) if pr > 0 else examples
    for model_name in MODELS:
        llm = build_llm(model_name, TEMPERATURE)
        with llm:
            for pt in PROMPT_TYPES:
                preds = []
                golds = []
                for i, ex in enumerate(poisoned):
                    corpus = build_corpus(ex, poisoned, distractor_pool_size=DISTRACTOR_POOL,
                                          seed=SEED + i, example_index=i)
                    retriever = Retriever(embedder=embedder, k=K)
                    retriever.build(corpus)
                    passages = retriever.retrieve(ex["claim"])
                    prompt = format_prompt(ex["claim"], passages, pt)
                    max_tok = {"standard": 64, "chain_of_thought": 512, "vigilant": 256}[pt]
                    response = llm.complete(prompt, max_tok)
                    preds.append(extract_label(response))
                    golds.append(ex["label"])

                # Split by gold label
                for lbl in ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
                    lbl_preds = [p for p, g in zip(preds, golds) if g == lbl]
                    lbl_golds = [g for g in golds if g == lbl]
                    if lbl_golds:
                        acc = compute_accuracy(lbl_preds, lbl_golds)
                        label_rows.append({
                            "model": MODEL_LABELS[model_name], "poison_rate": pr,
                            "prompt_type": pt, "gold_label": lbl,
                            "accuracy": acc, "n": len(lbl_golds),
                        })

embedder.close()
df_label = pd.DataFrame(label_rows)

# Show NEI accuracy at 0% and 50%
nei = df_label[df_label["gold_label"] == "NOT ENOUGH INFO"]
nei_pivot = nei.pivot_table(index=["model", "prompt_type"], columns="poison_rate", values="accuracy")
nei_pivot.columns = [f"{c:.0%}" for c in nei_pivot.columns]
print("=== Accuracy on NOT ENOUGH INFO claims only ===")
print("   (1 - this value approximates hallucination rate)")
print(nei_pivot.to_string(float_format="{:.3f}".format))

print("\n=== Accuracy on factual claims (SUPPORTS + REFUTES) ===")
factual = df_label[df_label["gold_label"] != "NOT ENOUGH INFO"]
factual_agg = factual.groupby(["model", "prompt_type", "poison_rate"])["accuracy"].mean().reset_index()
fac_pivot = factual_agg.pivot_table(index=["model", "prompt_type"], columns="poison_rate", values="accuracy")
fac_pivot.columns = [f"{c:.0%}" for c in fac_pivot.columns]
print(fac_pivot.to_string(float_format="{:.3f}".format))

## 10. Self-Consistency Summary Bar Chart

All conditions at poison_rate=0.5. Attribution: Wang et al. 2022 (cited in Zhou 2024 §2.1).

In [ ]:
n_models  = len(MODELS)
bar_width = 0.14
x         = np.arange(len(PROMPT_TYPES))
offsets   = np.linspace(-(n_models - 1) / 2, (n_models - 1) / 2, n_models) * bar_width

fig, ax = plt.subplots(figsize=(10, 4))
for i, model in enumerate(MODELS):
    sub  = df_sc[df_sc["model"] == model].set_index("prompt_type")
    vals = [sub.loc[pt, "self_consistency"] for pt in PROMPT_TYPES]
    ax.bar(x + offsets[i], vals, bar_width,
           label=MODEL_LABELS[model],
           color=MODEL_COLORS[model],
           edgecolor="white", linewidth=0.5, alpha=0.88)

ax.set_title(f"Self-Consistency (poison_rate=0.5, temp={TEMP_SC}, sc_runs={SC_RUNS}, N={N_SC})",
             fontsize=11, pad=8)
ax.set_xticks(x)
ax.set_xticklabels(PROMPT_TYPES, fontsize=9)
ax.set_ylim(0.6, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", linestyle=":", alpha=0.4)
ax.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig("../figures/05_self_consistency.png", dpi=150, bbox_inches="tight")
print("Saved -> figures/05_self_consistency.png")
plt.show()

## 11. Report-Ready Summary Tables

Compact pivot tables for inclusion in the final report.

In [ ]:
# Table 1: Accuracy pivot (model x prompt rows, poison_rate columns)
pivot_acc = df_main.pivot_table(index=["model", "prompt_type"], columns="poison_rate", values="accuracy")
pivot_acc.columns = [f"{c:.0%}" for c in pivot_acc.columns]
pivot_acc.index = pivot_acc.index.map(lambda x: (MODEL_LABELS[x[0]], x[1]))
print("=== Table 1: Accuracy by Condition ===")
print(pivot_acc.to_string(float_format="{:.3f}".format))

# Table 2: Macro-F1 pivot
pivot_f1 = df_main.pivot_table(index=["model", "prompt_type"], columns="poison_rate", values="macro_f1")
pivot_f1.columns = [f"{c:.0%}" for c in pivot_f1.columns]
pivot_f1.index = pivot_f1.index.map(lambda x: (MODEL_LABELS[x[0]], x[1]))
print("\n=== Table 2: Macro-F1 by Condition ===")
print(pivot_f1.to_string(float_format="{:.3f}".format))

# Table 3: Hallucination rate pivot
pivot_hall = df_main.pivot_table(index=["model", "prompt_type"], columns="poison_rate", values="hallucination_rate")
pivot_hall.columns = [f"{c:.0%}" for c in pivot_hall.columns]
pivot_hall.index = pivot_hall.index.map(lambda x: (MODEL_LABELS[x[0]], x[1]))
print("\n=== Table 3: Hallucination Rate by Condition ===")
print(pivot_hall.to_string(float_format="{:.3f}".format))

# Table 4: Self-consistency (single poison_rate=0.5)
sc_table = df_sc.copy()
sc_table["model"] = sc_table["model"].map(MODEL_LABELS)
sc_table = sc_table.set_index(["model", "prompt_type"])
print(f"\n=== Table 4: Self-Consistency (poison_rate=0.5, temp={TEMP_SC}, sc_runs={SC_RUNS}) ===")
print(sc_table.to_string(float_format="{:.3f}".format))

## 12. Key Findings

Mean accuracy values below are averaged across all prompt types and poison rates unless noted.
Model ranking: **Phi-3.5 (0.587) > Gemma-2 (0.505) > Llama-3.2 (0.441) > Qwen2.5 (0.433) > SmolLM2 (0.379)**.

---

### F1: Accuracy degrades monotonically; no U-shaped recovery
Unlike encoder-decoder models (e.g. flan-t5), all five decoder-only instruction-tuned models show
largely monotonic accuracy degradation as poison rate increases. At `poison_rate=1.0` (P@k=0),
models do not fall back on confident parametric knowledge: they instead default to
**NOT ENOUGH INFO**, which is often correct for NEI claims but wrong for factual ones.
The partial recovery at 100% poisoning reported for earlier models does not generalise here.
Attribution: Lewis et al. 2020 (parametric vs contextual knowledge); failure to replicate
Zhou et al. 2024 robustness pattern on stronger instruction-tuned models.

### F2: Phi-3.5 is the top performer; SmolLM2 is the weakest
Phi-3.5 leads by a large margin (mean acc 0.587 vs Gemma-2 at 0.505, SmolLM2 at 0.379).
Phi-3.5 at `poison_rate=0.0` reaches **0.720** with both CoT and vigilant prompts: the highest
single-condition accuracy in the grid. SmolLM2 plateaus at ~0.340 on standard regardless of
poison rate, suggesting it relies almost entirely on default parametric priors and ignores
retrieved context.
Attribution: Zhou et al. 2024: multi-model comparison (§4).

### F3: Chain-of-thought is the best prompt type on average
Mean accuracy: CoT **0.519** > standard **0.482** > vigilant **0.406** (averaged over all 5 models
and all poison rates). CoT benefits Phi-3.5 and Llama-3.2 the most (+5–7pp vs standard).
Attribution: Zhou et al. 2024 §2.1 (CoT as robustness technique: confirmed here).

### F4: Vigilant prompt hurts four out of five models
Qwen2.5 (0.340), SmolLM2 (0.340), and Llama-3.2 (0.368) all drop sharply under the vigilant
prompt, performing near-random. Only **Phi-3.5 benefits** (vigilant acc 0.580 ≈ CoT 0.616),
suggesting that structured consistency-check instructions require strong instruction-following
capability to work as intended. The vigilant prompt is not a robust defence strategy for
smaller models.
Attribution: Zhou et al. 2024 §2.1/§3 (vigilant prompting).

### F5: SmolLM2/vigilant has catastrophically high hallucination rate (~0.71)
SmolLM2 under the vigilant prompt systematically asserts unsupported facts regardless of poison
rate, with hallucination rates around **0.706–0.765** across all conditions. This is the highest
value in the entire grid and signals that the vigilant prompt's cross-checking step actively
misleads a model that cannot reason reliably.
Phi-3.5/standard shows moderately high hallucination at 0% poisoning (0.588) but it decreases
as poison rate increases: the opposite of the degradation pattern, consistent with F1 above.
Attribution: Zhou et al. 2024: Factuality dimension.

### F6: Hallucination decreases with poison rate (consistent across all models)
At 100% poisoning (P@k=0), retrieved passages contain no gold evidence → models answer
NOT ENOUGH INFO by default → hallucination (false assertions on NEI claims) drops toward zero.
The **most dangerous regime remains 0–50% poisoning**, where models partially trust corrupted
context and over-assert. This pattern is robust across all 5 models.
Attribution: Zhou et al. 2024: Factuality dimension.

### F7: Self-consistency is low for Llama-3.2/CoT and SmolLM2
Range: 0.680 (Llama-3.2/CoT) to 0.893 (Gemma-2/standard). Llama-3.2 and SmolLM2 under CoT
show the highest output variability under passage-order perturbation, suggesting that
step-by-step reasoning in weaker models is sensitive to context ordering.
Gemma-2/standard (0.893) and Qwen2.5/vigilant (0.827) are the most consistent.
Attribution: Wang et al. 2022 (self-consistency); Zhou et al. 2024 §2.1.

### F8: Prompt engineering gains are model-capability-dependent
The benefit of advanced prompt templates (CoT, vigilant) scales with model quality.
For SmolLM2 and Qwen2.5, no prompt template significantly outperforms the others in robustness.
For Phi-3.5, both CoT and vigilant provide consistent gains (+7–10pp vs standard at 0% poisoning).
This suggests that prompt engineering is not a universal mitigation strategy for retrieval poisoning
but requires a minimum level of instruction-following capability to be effective.
Attribution: Singal et al. 2024 §4; Zhou et al. 2024 §2.1.

### Summary
Under retrieval poisoning with decoder-only instruction-tuned models, **CoT is the most robust
prompt strategy overall**, while the vigilant prompt is only beneficial for Phi-3.5.
Accuracy degrades monotonically with poison rate: no U-shaped recovery.
The 0–50% poisoning regime remains the most dangerous. Phi-3.5 is the best-performing model
at every poison rate and with every prompt type.